# Redox FHIR Pipeline - Bundle Metadata

This notebook extracts bundle-level metadata from Redox FHIR bundles, including Redox-specific Meta information.

## Tables Created
| Table | Description |
|-------|-------------|
| `bundle_meta` | Bundle-level metadata with primary key |

## Redox Meta Structure
Redox bundles include a top-level `Meta` object (note: capitalized) containing:
- `profile`: Array of profile URLs
- Additional Redox-specific metadata fields

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE schema_use STRING DEFAULT 'bronze';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE schema_use = COALESCE(:schema_use, schema_use);

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog() AS catalog, current_schema() AS schema;

## Create Bundle Metadata Table

This table serves as the central reference for all FHIR bundles, with a primary key that other resource tables can reference via foreign key.

In [ ]:
CREATE OR REFRESH STREAMING TABLE bundle_meta (
  bundle_uuid STRING NOT NULL PRIMARY KEY COMMENT 'Unique bundle identifier'
  ,ingest_time TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp'
  ,file_metadata STRUCT<
    file_path: STRING,
    file_name: STRING,
    file_size: BIGINT,
    file_block_start: BIGINT,
    file_block_length: BIGINT,
    file_modification_time: TIMESTAMP
  > NOT NULL COMMENT 'Source file metadata'
  ,resource_type STRING COMMENT 'FHIR resourceType (should be Bundle)'
  ,bundle_type STRING COMMENT 'Bundle type (collection, transaction, etc.)'
  ,bundle_id STRING COMMENT 'FHIR Bundle ID'
  ,bundle_timestamp STRING COMMENT 'Bundle timestamp'
  ,entry_count INT COMMENT 'Number of entries in the bundle'
  ,redox_meta VARIANT COMMENT 'Redox-specific Meta object'
  ,redox_profile VARIANT COMMENT 'Redox profile URLs'
)
COMMENT 'Bundle-level metadata for Redox FHIR bundles'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'bronze'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS SELECT
  bundle_uuid
  ,ingest_time
  ,file_metadata
  ,fhir:resourceType::STRING AS resource_type
  ,fhir:type::STRING AS bundle_type
  ,fhir:id::STRING AS bundle_id
  ,fhir:timestamp::STRING AS bundle_timestamp
  ,COALESCE(size(variant_get(fhir, '$.entry')::ARRAY<VARIANT>), 0) AS entry_count
  ,fhir:Meta AS redox_meta
  ,fhir:Meta.profile AS redox_profile
FROM STREAM fhir_bronze_variant;

In [ ]:
-- Verify bundle metadata
SELECT 
  bundle_uuid,
  bundle_id,
  bundle_type,
  bundle_timestamp,
  entry_count,
  file_metadata.file_name,
  ingest_time
FROM bundle_meta
ORDER BY ingest_time DESC
LIMIT 10;

In [ ]:
-- Analyze Redox profiles in use
SELECT 
  redox_profile,
  COUNT(*) AS bundle_count
FROM bundle_meta
GROUP BY redox_profile
ORDER BY bundle_count DESC;

In [ ]:
-- Summary statistics
SELECT
  COUNT(*) AS total_bundles,
  SUM(entry_count) AS total_entries,
  AVG(entry_count) AS avg_entries_per_bundle,
  MIN(ingest_time) AS first_ingest,
  MAX(ingest_time) AS last_ingest
FROM bundle_meta;